# chromosomix in a notebook

The [chromosomix](https://chromosomix.app/) viewer as a Jupyter widget: the app itself in a cell — the
map, its tracks, its side panels and its gestures — driven from Python, and telling Python where the
reader has taken it.

This notebook opens the repository's own yeast demo: a Micro-C map of *S. cerevisiae* with ATAC-seq,
MNase-seq and the genes beside it, and a 3D model of the genome. The files are on this disk, so the
kernel serves them to the viewer by byte range; an address — `https://`, `s3://`, a Dropbox or Drive
share — is opened where it is, as in the app.

In [ ]:
from pathlib import Path

from chromosomix import Chromosomix

DEMO = Path('../../demo/yeast').resolve()
MAP = DEMO / 'Yeast_microC_G2M_1kb.cool'
sorted(path.name for path in DEMO.iterdir())[:12]

## A map and its tracks

`region` frames the map, `tracks` are put beside it once it is drawn, and every other keyword is a key
the app's address bar carries — here, counts on a log₁₀ scale.

In [ ]:
view = Chromosomix(
    MAP,
    region='IV:1000000-1400000',
    tracks=[DEMO / 'Yeast_ATAC.bw', DEMO / 'Yeast_MNase.bw', DEMO / 'Yeast_annotations.gtf'],
    transform='log10',
    height=860,
)
view

## Where the reader is

`view.state` is the app's view state, as its address bar carries it, and it follows the reader: pan or
zoom the map above and run the next cell again. It is a traitlet, so a cell can also be told.

In [ ]:
import ipywidgets as widgets

where = widgets.HTML()


def follow(change):
    state = change['new'] or {}
    where.value = f"<code>{state.get('region', state.get('view', ''))}</code> · {state.get('transform', '')}"


view.observe(follow, names='state')
follow({'new': view.state})
where

## Driving it from Python

`goto` frames a region; `navigate` lays any patch over the view — here, the map as observed over
expected on a log₂ scale. The cell above follows both.

In [ ]:
view.goto('IV:1150000-1300000')

In [ ]:
view.navigate(transform='log2', oe=True)

`add` is the header's `+`: a track among the tracks, another map beside the map, or a 3D model of the
genome, which opens in a cell of its own beside the map, panned and zoomed with it.

In [ ]:
view.add(DEMO / 'Yeast_structure.g3d.gz')

## A 1D browser

With no map, the first file becomes the tab's: a bigWig opens a genome browser over the assembly it
declares, and the rest are its tracks. `goto` then moves its locus.

In [ ]:
browser = Chromosomix(
    tracks=[DEMO / 'Yeast_ATAC.bw', DEMO / 'Yeast_ChIP_Pol2-vs-input.bw', DEMO / 'Yeast_annotations.gtf'],
    height=540,
)
browser

In [ ]:
browser.goto('IV:400,001-460,000')

## Remote files, and which viewer

An address needs nothing from the kernel:

```python
Chromosomix(
    'https://rsglab-data.s3.eu-west-3.amazonaws.com/demo/yeast/Yeast_microC_G2M_1kb.cool',
    region='IV:1000000-1400000',
    tracks=['https://rsglab-data.s3.eu-west-3.amazonaws.com/demo/yeast/Yeast_ATAC.bw'],
)
```

The frame holds https://chromosomix.app/ unless `viewer=` names another build: a release
(`viewer='0.17.0'`, kept for ever, which is what a notebook that must draw the same picture next year
names), any address, or a built `dist/` on this disk, served beside the data. With files on this disk,
`chromosomix.install_viewer()` fetches the npm package's build once and every widget after uses it, so
the app and the data are one origin and the notebook works offline. From a clone, `npm run
build:package` and `CHROMOSOMIX_VIEWER=dist/` do the same.

`view.viewer` is the build this widget frames, and `view.version` the version that build says it is
— once its frame is up, so it is empty in a notebook run whole before the frames have drawn.

In [ ]:
view.viewer